# Roman Strong Lens — Blind Holdout Evaluation (ViT Classifier + CNN Regressor)

This notebook is a **hybrid** of the two single-architecture holdout notebooks: it runs a
**Vision Transformer as the Stage 1 classifier** (`TransformerNet`, from
`roman_lens_vit_two_stage_pipeline_6040.ipynb`) and a **CNN as the Stage 2 regressor**
(`ResNetCNN`, from `roman_lens_two_stage_pipeline.ipynb`). Everything else — data loading,
cascade logic, evaluation, worst-prediction inspection, and quartile binning — mirrors the
two single-architecture holdout notebooks.

The dataset's `.h5` file has no `theta_e` ground truth — the true Einstein radii live in a
separate `.csv` instead.

### Pipeline overview
1. Rebuild both architectures: `TransformerNet` (ViT, classifier) and `ResNetCNN` (CNN, regressor)
2. Load each model's own saved weights
3. Load every image in the new `.h5` file (blind — no filtering by any true-label attribute)
4. Run the cascade: ViT classifies lens vs. nonlens, then the CNN regresses θ_e only on predicted lenses
5. Load the `.csv` of true answers and join it to the predictions on `uid`
6. Score Stage 1 (accuracy/precision/recall/F1) and Stage 2 (RMSE/MAE/R²)
7. Worst predictions, quartile-binned error, and a tightened cascade gated on correctly-classified images only

**Self-contained** — redefines both model classes and helper functions so it doesn't need to
run in the same kernel as either training notebook. Only needs the ViT classifier weights and
the CNN regressor weights present on disk.

### My best results were on an 60/40 CNN and a 1/99 ViT

### Note on SNR
As with both single-architecture holdout notebooks, this dataset's `.h5`/`.csv` don't carry an
`snr` value anywhere, so the error-vs-SNR analysis is omitted here.

---
## Cell 1: Imports

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.visualization import make_lupton_rgb

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

from sklearn.metrics import r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU name: {torch.cuda.get_device_name(0)}')

---
## Cell 2: Rebuild Both Model Architectures

Two independent architectures, one per stage:
- **`TransformerNet`** (`PatchEmbedding` + `nn.TransformerEncoder`) — used as the **classifier** (Stage 1)
- **`ResNetCNN`** (`ResidualBlock` stack) — used as the **regressor** (Stage 2)

Both class definitions must match the originals exactly, or `load_state_dict` will fail — the
saved state dicts only contain weights, not the class definitions.

In [ ]:
class PatchEmbedding(nn.Module):
    '''
    Small convolutional stem + patchify, instead of a single linear
    projection straight from raw pixels.

    The stem (stride-1, 3x3 convs) lets the model build up basic local
    texture/edge sensitivity before the image gets chopped into patches
    and handed to self-attention. Spatial size is preserved through the
    stem (stride=1, padding=1), so padding-to-patch-multiple happens
    after the stem instead of directly on the raw image.
    '''
    def __init__(self, in_channels=3, patch_size=9, embed_dim=128,
                 img_size=73, stem_channels=32):
        super().__init__()
        self.patch_size = patch_size

        # ── Convolutional stem: 3 stride-1 conv layers, no downsampling ──
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, stem_channels, kernel_size=3,
                      stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(stem_channels, stem_channels, kernel_size=3,
                      stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(stem_channels, stem_channels, kernel_size=3,
                      stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.ReLU(inplace=True),
        )

        # Compute padded size (next multiple of patch_size >= img_size)
        self.padded_size = math.ceil(img_size / patch_size) * patch_size
        pad_total = self.padded_size - img_size
        self.pad_left = pad_total // 2
        self.pad_right = pad_total - self.pad_left

        self.grid_size = self.padded_size // patch_size
        self.num_patches = self.grid_size * self.grid_size

        # Patchify: projects from stem_channels (not raw 3) to embed_dim
        self.proj = nn.Conv2d(stem_channels, embed_dim,
                               kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.stem(x)              # (batch, stem_channels, H, W)
        x = F.pad(x, (self.pad_left, self.pad_right,
                       self.pad_left, self.pad_right))
        x = self.proj(x)              # (batch, embed_dim, grid, grid)
        x = x.flatten(2)              # (batch, embed_dim, num_patches)
        x = x.transpose(1, 2)         # (batch, num_patches, embed_dim)
        return x


class TransformerNet(nn.Module):
    '''
    Vision Transformer — used here as the Stage 1 CLASSIFIER only.
    Final layer is a single raw number (Linear(64, 1), no activation),
    treated as a logit and passed through BCEWithLogitsLoss/sigmoid.

    Architecture:
        Input        : (batch, 3, 91, 91)
        Patch embed  : conv stem -> pad to grid multiple -> 9x9 patches -> embed_dim
        CLS token    : learnable token prepended to the sequence
        Pos embed    : learnable positional embedding added to every token
        Transformer  : `depth` stacked encoder blocks (self-attn + MLP)
        Head         : CLS token's output -> MLP -> scalar
    '''
    def __init__(self, in_channels=3, img_size=73, patch_size=9,
                 embed_dim=96, depth=4, num_heads=6, mlp_ratio=4.0,
                 dropout=0.1):
        super().__init__()

        self.patch_embed = PatchEmbedding(in_channels, patch_size, embed_dim, img_size)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True   # pre-LN blocks train more stably than post-LN
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

        self.fc_head = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 1)
        )

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        batch_size = x.shape[0]

        x = self.patch_embed(x)                       # (batch, num_patches, embed_dim)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)          # (batch, num_patches+1, embed_dim)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        x = self.transformer(x)
        x = self.norm(x)

        cls_output = x[:, 0]                           # take CLS token only
        out = self.fc_head(cls_output)
        return out.squeeze(1)

In [ ]:
class ResidualBlock(nn.Module):
    """
    A single residual block: two 3x3 convs with batch norm and ReLU,
    plus a skip connection that adds the block's input back to its output.
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                                stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)
        return out


class ResNetCNN(nn.Module):
    """
    Residual CNN — used here as the Stage 2 REGRESSOR only.
    Final layer is a single raw number (Linear(64, 1), no activation) —
    that number IS the predicted theta_e.
    """
    def __init__(self):
        super(ResNetCNN, self).__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )

        self.stage1 = nn.Sequential(
            ResidualBlock(16, 32, stride=2),
            ResidualBlock(32, 32, stride=1)
        )
        self.stage2 = nn.Sequential(
            ResidualBlock(32, 64, stride=2),
            ResidualBlock(64, 64, stride=1)
        )
        self.stage3 = nn.Sequential(
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128, stride=1)
        )
        self.stage4 = nn.Sequential(
            ResidualBlock(128, 256, stride=2),
            ResidualBlock(256, 256, stride=1)
        )

        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.fc_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.global_pool(x)
        x = self.fc_head(x)
        return x.squeeze(1)

---
## Cell 3: Load the Trained Weights

**Update these paths.** The classifier is the ViT (`TransformerNet`) — point it at the same
`best_clf_model_vit*.pt` file used in the ViT holdout notebook. The regressor is the CNN
(`ResNetCNN`) — point it at the same `best_model*.pt` file used in the CNN holdout notebook.
These are two different training runs with two different weight files; there's no requirement
that they came from the same class-balance split, but keeping that consistent (e.g. both from
the 60/40 run) makes the comparison to the single-architecture holdout notebooks cleanest.

The ViT needs its architecture hyperparameters passed explicitly (`img_size=91, patch_size=9,
embed_dim=96, depth=4, num_heads=6, mlp_ratio=4.0, dropout=0.1`, matching training). The CNN
has no such hyperparameters to set — it's fully convolutional with global average pooling, so
it doesn't care about spatial input size.

In [ ]:
# ── UPDATE THESE PATHS ──────────────────────────────────────────────────────
clf_weights_path = 'best_clf_model_vit.pt'   # ViT classifier (Stage 1)
reg_weights_path = 'best_model.pt'           # CNN regressor  (Stage 2)
# ─────────────────────────────────────────────────────────────────────────────

VIT_KWARGS = dict(in_channels=3, img_size=91, patch_size=9,
                   embed_dim=96, depth=4, num_heads=6, mlp_ratio=4.0, dropout=0.1)

clf_model = TransformerNet(**VIT_KWARGS).to(device)
clf_model.load_state_dict(torch.load(clf_weights_path, map_location=device))
clf_model.eval()

model = ResNetCNN().to(device)
model.load_state_dict(torch.load(reg_weights_path, map_location=device))
model.eval()

print('Loaded ViT classifier weights from:', clf_weights_path)
print('Loaded CNN regressor weights from :', reg_weights_path)

---
## Cell 4: Load the New Dataset (Blind — No `theta_e` in the `.h5`)

Same `normalize`/`get_filter_dataset` helpers as both single-architecture holdout notebooks.
Pulls **every** entry — lens and nonlens alike, with no filtering — because the whole point is
to run the entire blind set through both stages of the cascade. Also pulls out each system's
`uid` to join predictions back to the CSV's true answers afterward.

In [ ]:
def normalize(img):
    mn, mx = img.min(), img.max()
    if mx - mn == 0:
        return np.zeros_like(img, dtype=np.float32)
    return ((img - mn) / (mx - mn)).astype(np.float32)

def attr_val(attrs, key):
    """Attrs in this dataset are stored as [value, description] — grab the value."""
    v = attrs[key]
    if isinstance(v, np.ndarray) and v.shape == (2,):
        return v[0]
    return v

def get_filter_dataset(group, filter_name):
    matches = [k for k in group.keys() if k.endswith(f'_{filter_name}')]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one key ending in _{filter_name}, found {matches}")
    return group[matches[0]]


# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
holdout_h5_filepath = 'roman_data_challenge_rung_0_6040split_unlabeled_v_1_0.h5'
# ─────────────────────────────────────────────────────────────────────────────

f_holdout = h5py.File(holdout_h5_filepath, 'r')
images_group_holdout = f_holdout['images']

holdout_images = []
holdout_uids = []

for strong_lens_group in images_group_holdout.keys():
    group = images_group_holdout[strong_lens_group]

    uid = str(attr_val(group.attrs, 'uid'))

    ds_r = get_filter_dataset(group, 'F158')
    ds_g = get_filter_dataset(group, 'F129')
    ds_b = get_filter_dataset(group, 'F106')

    norm_r = normalize(ds_r[:])
    norm_g = normalize(ds_g[:])
    norm_b = normalize(ds_b[:])

    stacked = np.stack([norm_r, norm_g, norm_b], axis=0)
    holdout_images.append(stacked)
    holdout_uids.append(int(uid))

f_holdout.close()

holdout_images = np.array(holdout_images, dtype=np.float32)
holdout_uids = np.array(holdout_uids, dtype=np.int64)

print(f'Holdout images shape : {holdout_images.shape}')
print(f'Holdout uids shape   : {holdout_uids.shape}')
print(f'uid range             : {holdout_uids.min()} to {holdout_uids.max()}')

---
## Cell 5: Load the CSV of True Answers

In [ ]:
# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
holdout_csv_filepath = 'roman_data_challenge_rung_0_6040split_answer_key_v_1_0.csv'
# ─────────────────────────────────────────────────────────────────────────────

truth_df = pd.read_csv(holdout_csv_filepath)
truth_df = truth_df.set_index('uid')

print(f'Truth CSV shape: {truth_df.shape}')
truth_df.head()

---
## Cell 6: Run the Two-Stage Cascade (ViT Classifier → CNN Regressor)

Every image goes through `clf_model` (the ViT) first, and only images it classifies as lenses
get passed to `model` (the CNN) for a θ_e prediction. Batched here since the holdout set is
run in one shot rather than via a `DataLoader` during training.

In [ ]:
def run_two_stage_pipeline(images_tensor, clf_model, reg_model, device, threshold=0.5, batch_size=256):
    """
    Stage 1 (ViT): classify every image as lens/nonlens.
    Stage 2 (CNN): only images classified as lenses get passed through the regressor.
    Returns predicted is_lens (0/1 per image) and predicted theta_e
    (NaN for images the classifier called nonlens — they never reach Stage 2).
    Processes in batches to avoid loading the whole holdout set onto the GPU at once.
    """
    clf_model.eval()
    reg_model.eval()

    all_is_lens = []
    all_theta_e = []

    with torch.no_grad():
        for start in range(0, len(images_tensor), batch_size):
            batch = images_tensor[start:start + batch_size].to(device)

            logits = clf_model(batch)
            probs = torch.sigmoid(logits)
            pred_is_lens = (probs > threshold).float()

            pred_theta_e = torch.full((len(batch),), float('nan'), device=device)
            lens_mask = pred_is_lens == 1
            if lens_mask.any():
                pred_theta_e[lens_mask] = reg_model(batch[lens_mask])

            all_is_lens.append(pred_is_lens.cpu().numpy())
            all_theta_e.append(pred_theta_e.cpu().numpy())

    return np.concatenate(all_is_lens), np.concatenate(all_theta_e)


holdout_tensor = torch.tensor(holdout_images, dtype=torch.float32)

pred_is_lens, pred_theta_e = run_two_stage_pipeline(holdout_tensor, clf_model, model, device)

print(f'Ran pipeline on {len(holdout_images)} holdout images')
print(f'Stage 1 (ViT) classified {int(pred_is_lens.sum())} as lenses, '
      f'{len(pred_is_lens) - int(pred_is_lens.sum())} as nonlens')

---
## Cell 7: Join Predictions to the True Answers (by `uid`)

The CSV's `deflector_only` flag mirrors the `.h5` attribute of the same name:
`deflector_only=True` means no lensing occurred, i.e. **not** a strong lens. So true
`is_lens = ~deflector_only`. `einstein_radius` is the true θ_e.

In [ ]:
true_deflector_only = truth_df.loc[holdout_uids, 'deflector_only'].values
true_is_lens = (~true_deflector_only).astype(np.float32)
true_theta_e = truth_df.loc[holdout_uids, 'einstein_radius'].values.astype(np.float32)

print(f'Matched {len(holdout_uids)} holdout predictions to CSV rows')
print(f'True lens count: {int(true_is_lens.sum())} / {len(true_is_lens)}')

---
## Cell 8: Evaluate — Stage 1 (ViT) and Stage 2 (CNN)

In [ ]:
acc  = accuracy_score(true_is_lens, pred_is_lens)
prec = precision_score(true_is_lens, pred_is_lens)
rec  = recall_score(true_is_lens, pred_is_lens)
f1   = f1_score(true_is_lens, pred_is_lens)
cm   = confusion_matrix(true_is_lens, pred_is_lens)

print('── Stage 1: ViT Classifier Performance (Holdout) ───────────────────')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1        : {f1:.4f}')
print('Confusion matrix (rows=true, cols=pred), order = [nonlens, lens]:')
print(cm)

true_lens_mask = true_is_lens == 1
pred_lens_mask = pred_is_lens == 1
true_positive_mask = true_lens_mask & pred_lens_mask

n_true_lens = int(true_lens_mask.sum())
print(f'\n── Pipeline Breakdown ────────────────────────────────────────────')
print(f'True lenses in holdout set      : {n_true_lens}')
print(f'  → passed through to Stage 2  : {int(true_positive_mask.sum())} '
      f'({100*true_positive_mask.sum()/n_true_lens:.1f}%)')
print(f'  → missed at Stage 1          : {int((true_lens_mask & ~pred_lens_mask).sum())}')
print(f'Nonlens wrongly passed to Stage 2: {int(((~true_lens_mask) & pred_lens_mask).sum())}')

rmse = np.sqrt(np.mean((pred_theta_e[pred_lens_mask] - true_theta_e[pred_lens_mask]) ** 2))
mae  = np.mean(np.abs(pred_theta_e[pred_lens_mask] - true_theta_e[pred_lens_mask]))
r2   = r2_score(true_theta_e[pred_lens_mask], pred_theta_e[pred_lens_mask])

print(f'\n── Stage 2: CNN Regression Performance (Holdout, all images ViT called lenses) ──')
print(f'RMSE : {rmse:.4f} arcseconds')
print(f'MAE  : {mae:.4f} arcseconds')
print(f'R²   : {r2:.4f}')

---
## Cell 9: Predicted vs. True θ_e (Holdout)

A quick scatter plot against the CSV's true Einstein radii for the correctly-classified
lenses, with the y=x line for reference.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(true_theta_e[true_positive_mask], pred_theta_e[true_positive_mask],
            s=8, alpha=0.4, color='steelblue')

lims = [0, max(true_theta_e[true_positive_mask].max(), pred_theta_e[true_positive_mask].max()) * 1.05]
plt.plot(lims, lims, 'k--', linewidth=1, label='y = x')

plt.xlabel('True θ_e (arcseconds, from CSV)')
plt.ylabel('Predicted θ_e (arcseconds)')
plt.title('Holdout Set — Predicted vs. True Einstein Radius (ViT clf + CNN reg)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Cell 10: Worst Predictions — Combined Lupton RGB

One combined Lupton RGB image per system (F158/F129/F106), each panel's title carrying the
image name (`uid`), predicted θ_e, true θ_e, and percent error. Restricted to the
correctly-classified lenses (`true_positive_mask`) from Cell 8, since those are the only
holdout images with both a true θ_e and a Stage 2 (CNN) prediction to compare.

In [ ]:
def get_worst_predictions(preds, truths, uids, n_worst=25):
    """
    Returns a DataFrame of the n_worst predictions by absolute error, with
    columns for prediction, truth, error, and the system's uid.
    """
    abs_errors = np.abs(preds - truths)
    worst_order = np.argsort(abs_errors)[::-1][:n_worst]

    df = pd.DataFrame({
        'holdout_pos':  worst_order,
        'uid':          uids[worst_order],
        'pred':         preds[worst_order],
        'truth':        truths[worst_order],
        'abs_error':    abs_errors[worst_order],
        'signed_error': (preds - truths)[worst_order],
    })
    df['pct_error'] = 100 * df['abs_error'] / df['truth']
    return df


def print_worst_summary(worst_df, n_show=25):
    """Pretty-print the worst predictions table for quick eyeballing."""
    cols = ['uid', 'truth', 'pred', 'abs_error', 'pct_error']
    print(f'\n── Worst {n_show} Predictions by Absolute Error (Holdout) ─────────────')
    print(worst_df[cols].head(n_show).to_string(index=False, float_format='%.4f'))

    print(f'\nWorst-{n_show} true θ_e stats:')
    print(f"  mean={worst_df['truth'].head(n_show).mean():.3f}  "
          f"median={worst_df['truth'].head(n_show).median():.3f}  "
          f"min={worst_df['truth'].head(n_show).min():.3f}  "
          f"max={worst_df['truth'].head(n_show).max():.3f}")


def plot_worst_images(worst_df, images, n_show=12, cols=3, panel_size=5):
    """
    Displays the combined Lupton RGB image for the worst-N holdout systems,
    one panel per system.
    """
    n_show = min(n_show, len(worst_df))
    rows = int(np.ceil(n_show / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(panel_size * cols, panel_size * rows))
    axes = np.atleast_1d(axes).flatten()

    for i, (_, r) in enumerate(worst_df.head(n_show).iterrows()):
        pos = int(r['holdout_pos'])
        img = images[pos]  # channels: [0]=F158 (R), [1]=F129 (G), [2]=F106 (B)

        rgb = make_lupton_rgb(image_r=img[0], image_g=img[1], image_b=img[2],
                               stretch=0.2, Q=4)

        ax = axes[i]
        ax.imshow(rgb)
        ax.axis('off')

        image_name = f"strong_lens_{int(r['uid']):08d}"
        title = (f"{image_name}\n"
                 f"pred={r['pred']:.3f}\"  true={r['truth']:.3f}\"  "
                 f"err={r['pct_error']:.1f}%")
        ax.set_title(title, fontsize=10)

    for ax in axes[n_show:]:
        ax.axis('off')

    plt.suptitle('Worst Predictions — Combined Lupton RGB (F158/F129/F106)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


worst_df = get_worst_predictions(
    pred_theta_e[true_positive_mask],
    true_theta_e[true_positive_mask],
    holdout_uids[true_positive_mask],
    n_worst=25
)
print_worst_summary(worst_df, n_show=25)
plot_worst_images(worst_df, holdout_images[true_positive_mask], n_show=12, cols=3, panel_size=5)

---
## Cell 11: RMSE vs MAE Binned by True Einstein Radius

Same quartile-binning approach as both single-architecture holdout notebooks. SNR-based
binning is omitted for the reason noted at the top of this notebook.

In [ ]:
eval_preds = pred_theta_e[true_positive_mask]
eval_truth = true_theta_e[true_positive_mask]

theta_e_bin_edges = np.percentile(eval_truth, [0, 25, 50, 75, 100])
theta_e_bin_edges[-1] += 0.01
theta_e_bin_labels = ['Q1 (smallest θ_e)', 'Q2', 'Q3', 'Q4 (largest θ_e)']

print(f'Holdout correctly-classified θ_e range: {eval_truth.min():.3f}" to {eval_truth.max():.3f}"')
print(f'Quartile edges: {theta_e_bin_edges}')

print('\n── RMSE / MAE / R² by Einstein Radius Quartile (Holdout) ───────────────')
print(f'{"Bin":<20} {"Range (arcsec)":<20} {"N":>6} {"RMSE":>8} {"MAE":>8} {"R²":>8}')
print('─' * 75)

bin_results = []
for i in range(len(theta_e_bin_edges) - 1):
    lo, hi = theta_e_bin_edges[i], theta_e_bin_edges[i + 1]
    mask = (eval_truth >= lo) & (eval_truth < hi)
    n_in_bin = mask.sum()

    if n_in_bin < 5:
        print(f'{theta_e_bin_labels[i]:<20} {"":<20} {n_in_bin:>6}    (too few samples)')
        continue

    bin_preds = eval_preds[mask]
    bin_truth = eval_truth[mask]

    bin_rmse = np.sqrt(np.mean((bin_preds - bin_truth) ** 2))
    bin_mae  = np.mean(np.abs(bin_preds - bin_truth))
    bin_r2   = r2_score(bin_truth, bin_preds)

    range_str = f'{lo:.3f}–{hi:.3f}'
    bin_results.append({
        'bin': theta_e_bin_labels[i], 'range': range_str, 'n': n_in_bin,
        'rmse': bin_rmse, 'mae': bin_mae, 'r2': bin_r2
    })

    print(f'{theta_e_bin_labels[i]:<20} {range_str:<20} {n_in_bin:>6} {bin_rmse:>8.4f} {bin_mae:>8.4f} {bin_r2:>8.4f}')

print('─' * 75)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bins_plotted = [r['bin']  for r in bin_results]
r2_plotted   = [r['r2']   for r in bin_results]
rmse_plotted = [r['rmse'] for r in bin_results]
mae_plotted  = [r['mae']  for r in bin_results]

axes[0].bar(bins_plotted, r2_plotted, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Einstein Radius Quartile')
axes[0].set_ylabel('R²')
axes[0].set_title('R² by Einstein Radius (θ_e) Quartile — Holdout (ViT clf + CNN reg)')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(alpha=0.3, axis='y')

x = np.arange(len(bins_plotted))
width = 0.35
axes[1].bar(x - width/2, rmse_plotted, width, label='RMSE', color='tomato')
axes[1].bar(x + width/2, mae_plotted,  width, label='MAE',  color='steelblue')
axes[1].set_xticks(x)
axes[1].set_xticklabels(bins_plotted, rotation=15)
axes[1].set_xlabel('Einstein Radius Quartile')
axes[1].set_ylabel('Error (arcseconds)')
axes[1].set_title('RMSE vs MAE by Einstein Radius (θ_e) Quartile — Holdout (ViT clf + CNN reg)')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## Cell 12: Cascade Re-Run — CNN Regressor Only Sees Correctly-Classified Lenses

Cell 6's cascade sends every image the ViT classifier *calls* a lens to the CNN regressor —
including false positives. This re-run tightens that gate: the regressor is only run on
images where the classifier's prediction **and** the true label agree the system is a lens.
False positives never reach Stage 2 at all here, isolating the CNN regressor's performance
from the ViT's Stage 1 mistakes.

Numerically this reproduces the same RMSE/MAE/R² as the `true_positive_mask` slice in Cell 8
— the regressor's predictions are deterministic per image — but here the gating happens
*before* inference rather than as a post-hoc filter, and we report how many images this saves
versus the looser Cell 6 gate.

In [ ]:
def run_pipeline_correctly_classified_only(images_tensor, clf_model, reg_model, true_is_lens,
                                            device, threshold=0.5, batch_size=256):
    """
    Stage 1 (ViT): classify every image as lens/nonlens.
    Stage 2 (CNN): only run the regressor on images where the classifier's prediction
    AND the true label agree it's a lens — i.e. correctly-classified lenses only.
    Everything else (missed lenses, false positives) gets NaN for theta_e and
    never touches the regressor.
    """
    clf_model.eval()
    reg_model.eval()

    all_is_lens = []
    all_theta_e = []

    with torch.no_grad():
        for start in range(0, len(images_tensor), batch_size):
            batch = images_tensor[start:start + batch_size].to(device)
            batch_true_is_lens = torch.tensor(
                true_is_lens[start:start + batch_size], device=device
            )

            logits = clf_model(batch)
            probs = torch.sigmoid(logits)
            pred_is_lens = (probs > threshold).float()

            pred_theta_e = torch.full((len(batch),), float('nan'), device=device)
            correctly_classified_mask = (pred_is_lens == 1) & (batch_true_is_lens == 1)
            if correctly_classified_mask.any():
                pred_theta_e[correctly_classified_mask] = reg_model(batch[correctly_classified_mask])

            all_is_lens.append(pred_is_lens.cpu().numpy())
            all_theta_e.append(pred_theta_e.cpu().numpy())

    return np.concatenate(all_is_lens), np.concatenate(all_theta_e)


cc_pred_is_lens, cc_pred_theta_e = run_pipeline_correctly_classified_only(
    holdout_tensor, clf_model, model, true_is_lens, device
)

cc_scored_mask = ~np.isnan(cc_pred_theta_e)
n_sent_cell6 = int((pred_is_lens == 1).sum())  # from Cell 6 — every predicted lens, incl. false positives
n_sent_here  = int(cc_scored_mask.sum())

print(f'Cell 6 cascade sent {n_sent_cell6} images to the CNN regressor (all predicted lenses)')
print(f'This re-run sent only {n_sent_here} images to the CNN regressor (correctly-classified lenses only)')
print(f'Skipped {n_sent_cell6 - n_sent_here} false-positive images that never needed regression')

cc_rmse = np.sqrt(np.mean((cc_pred_theta_e[cc_scored_mask] - true_theta_e[cc_scored_mask]) ** 2))
cc_mae  = np.mean(np.abs(cc_pred_theta_e[cc_scored_mask] - true_theta_e[cc_scored_mask]))
cc_r2   = r2_score(true_theta_e[cc_scored_mask], cc_pred_theta_e[cc_scored_mask])

print(f'\n── Stage 2 Performance — Correctly-Classified-Only Gate ────────────────')
print(f'RMSE : {cc_rmse:.4f} arcseconds')
print(f'MAE  : {cc_mae:.4f} arcseconds')
print(f'R²   : {cc_r2:.4f}')
print(f'(Should match Cell 8\'s true_positive_mask numbers — RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f})')

---
## Summary

| Step | Description |
|---|---|
| Model rebuild | Recreates `TransformerNet` (ViT, classifier) and `ResNetCNN` (CNN, regressor); loads each one's own saved weights |
| Data loading | Opens the new `.h5` file, builds RGB images for **every** system (blind — no true-label filtering), keeping each `uid` |
| Cascade | ViT classifies every holdout image; CNN regresses θ_e only where the ViT called it a lens |
| Truth join | Loads the `.csv`, matches rows to predictions on `uid` |
| Evaluation | Accuracy/precision/recall/F1 for Stage 1 (ViT); RMSE/MAE/R² for Stage 2 (CNN) |
| Worst predictions | Combined Lupton RGB images for the worst 25 by absolute error, labeled with uid/pred/true/pct error |
| Binned error | RMSE/MAE/R² broken out by true θ_e quartile |
| Tightened cascade | CNN regressor re-run gated on correctly-classified images only, isolating Stage 2 performance from Stage 1 (ViT) errors |

### Why this combination
Neither original notebook mixes architectures across stages — the ViT notebook uses
`TransformerNet` for both, the CNN notebook uses `ResNetCNN` for both. This notebook swaps in
the ViT's classifier (as of the last recorded comparison, the ViT had the edge on Stage 1
precision/F1 on the 1%/99% split) paired with the CNN's regressor (which has generally shown
tighter RMSE/MAE than the ViT regressor). Swap `clf_weights_path`/`reg_weights_path` or the
model classes in Cell 3 if a different pairing is wanted.

### On SNR
The dataset doesn't carry an SNR value anywhere, so the error-vs-SNR plot and SNR-based
binning from the original notebooks aren't reproduced here.

### Run order
Run top to bottom once. No training happens here — this only loads existing weights.

### If Stage 1 recall looks worse than on the original test set
That's a real signal the new dataset differs in distribution from the Rung 0 training data —
worth checking `substructure_flag` in the CSV against the original training distribution
before concluding the model itself has degraded.